# DS/CMPSC 410 Section 1 Spring 2025

## Instructor: Professor John Yen
## TAs: Peng Jin and Jingxi Zhu
# Lab 2: MapReduce in Spark and Understand Error Messages
## The goals of this lab are for you to be able to
## - Read a text file (into an RDD of lines)
## - Parse an RDD of lines of input text into an RDD of list of tokens (using map)
## - Transform and Flatten the RDD of list of token into a key-value RDD (using map), so that we can count the number of occurances of token using reduce by key.
## - Use reduceByKey to aggregate the input key-value pairs into total count for each tokens.
## - Be able to use .take to show contents of an RDD.
## - Be able to understand the timing of an error message due to lazy evaluation.
## Total Number of Exercises: 
- Exercise 1: 10 points
- Exercise 2: 5 points
- Exercise 3: 5 points
- Exercise 4: 10 points
- Exercise 5: 10 points
- Exercise 6: 10 points
- Exercise 7: 10 points
- Exercise 8: 10 points
## Total Points: 70 points

# Due: midnight, January 26 (Sunday), 2025

# NOTE: If you have not installed PySpark, do not continue.  Follow the instructions for Lab2 (under Topic 2 Lab 2 module in Canvas) to install PySpark in your ICDS account first. Close this window for now.

# Proceed only after you have installed PySpark.

# The first thing we need to do in each Jupyter Notebook running pyspark is to import pyspark first.

In [2]:
import pyspark

# 1. SparkContext

## Once we import pyspark, we need to import an important object called "SparkContext".  
## Q: What is SparkContext? Why is it needed?
Ans: Every spark program needs a SparkContext object, which provides critical context (e.g., whether this Spark session runs in local mode or cluster mode) for the run-time environment of the Spark session. This contextual information is used in Spark statements such as `textFile` (for reading a text file) and `stop` (for terminating a Spark session).

**Learning Tip: Only one Spark context is allowed in a Spark session (whether it is local mode or cluster mode).  Therefore, DO NOT run `SparkContext` again after you create a spark context.  Doing so will result in Error Messages.  If you really want to start a new/clean Spark Context, execute `sc.stop()` (the last statement of this Notebook) to terminate the current Spark Context before you create a new one.**

In [3]:
from pyspark import SparkContext

## Once we have a spark context variable, we can execute spark codes. 
- The first parameter of SparkContext specifies whether this Spark session is running in "local" mode or a cluster mode.  In this and future labs that run Spark in local mode, we use "local" as the value of the first parameter. When we prepare Spark code for cluster mode in later labs, we will need to modify this parameter value.
- The second parameter specifies a name for your Spark session.
- In creating the Spark Context variable below, we specified that this spark code is running in a 
`local` mode, with a name `Lab2`.
- After you run the cell below, be patient to wait for its completion before you "run" the next cell. When the left of the cell shows 
`[*]:`, it means the 'run' is not completed yet. The completion of running a cell is indicated by a number in the brackets such as
`[3]:`.
**You MUST wait for a cell to complete before you 'run' the next cell in Jupyter notebook. Otherwise, the execution of your later cells may generate errors because the inputs they need are not available yet (e.g., been not been generated by a previous PySpark statement that is still running).**

In [4]:
sc=SparkContext("local", "Lab2")
sc

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/23 12:50:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local appName=Lab2>

# The following statement reduces the amount of warning messages.

In [5]:
sc.setLogLevel("WARN")

## Exercise 1 (10 points) (a) Add your name below AND (b) replace ??? in the input path for the two `textFile` statements with your PSU Access ID.
## Answer for Exercise 1
- a: Aidan Vesci: ajv5723

# 2. Read Big Data Using sc.textFile (a transformation)
- Spark can read from a single file or a distributed file (which we will see in later labs) of big data using the method `textFile`.
- Reading data in spark requires (1) an active Spark Context, and (2) a path for the file to be red.
- The result of executing `sc.textFile("...")` is the creation of a template (called RDD) for storing (big) data from the file in a distributed/partitioned way.  
- However, the actual reading of the file is NOT immediate.  In stead, this is DELAYED until a Spark statement that demands immediate execution results. These Spark statements that demand execution results are called `actions`.  In contrast, Spark statements whose execution can be delayed are called `transformations`.   

In [6]:
text_RDD = sc.textFile("/storage/home/ajv5723/work/Lab2/HostageReleased.txt")
text_RDD

/storage/home/ajv5723/work/Lab2/HostageReleased.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

## Specifying minimum partitions of RDD generated from textFile
- An optional parameter of `textFile` is minPartitions, which specify the minimum number of partitions (even though still ONE RDD object) in which data red from the text file is stored.

In [7]:
text_RDD2 = sc.textFile("/storage/home/ajv5723/work/Lab2/HostageReleased.txt", minPartitions = 2)

In [8]:
text_RDD2

/storage/home/ajv5723/work/Lab2/HostageReleased.txt MapPartitionsRDD[3] at textFile at NativeMethodAccessorImpl.java:0

# 3. RDD (template or data/intermediate results)
- RDD is the primary distributed data structure used by Spark for generating/accessing big data and big intermediate results (like the key-value pairs used by MapReduce for counting Document Frequency) in a cluster.  
- We will talk more about RDD next week. For now, we can view RDD (conceptually) as a BIG LIST.
- When we use `textFile` of Spark to read an input file, it returns an RDD (a big list), where each entry in the big table is a STRING that reprsents a LINE of the input file. However, as we mentioned before, **the actual content of the RDD is not available until a subsequent action is executed**. Before then, the RDD only contains **the template** for generating the actual content (as we will discuss next week).
- Some contents of RDD can be forced to show using `.take()` method, which is an action.

# 4. Take (an action)
- Take is an action, which forces immediate generation of the content of input RDD.  In another word, it triggers "lazy evaluation" of preceecing transformation steps.  We will elaborate on this next week.
- Applying `take(n)` method on an RDD forces the first n entries of the RDD to be shown. 
- Learning Tip: **Using take with a small parameter to show a few elements of an RDD is an excellent programming practice in the local mode.  This enables you to detect errors by noticing and analyzing unexpected/suspecious results.**

# Exercise 2 (5%)
Complete the code below to show the first FIVE lines of the input text, regardless whether the line contain text or not.

In [9]:
text_RDD.take(5)

[' “We can breathe a little more again.” Israelis rejoice at release of hostages',
 'From CNN’s Nadeen Ebrahim and Mike Schwartz in Tel Aviv ',
 '',
 ' Israelis were elated following the release of three hostages from Gaza on Sunday, saying it almost feels unreal after so many months of waiting.',
 '']

# 5 Parsing Text Into Tokens using Map (a transformation)
- Map of Spark applies a function to an input RDD. We often use a `lambda` expression to describe an unnamed function as the parameter for map. Map in Spark is analogous to the map step in MapReduce
- The input RDD to the `map` statement below can be viewed for now (we will elaborate on RDD later) as a **big list**.
- The body of the lambda expression 'x.strip().split(" ")' applies Python `strip` and `split` methods to the input parameter of the lambda expression (i.e., x).
- The Python `strip` method for strings removes spaces at the beginning and at the end of a string.
- The Python `split` method for strings split the string using the specified delimiter (which is space `" "` in this case).
- Map statements returns an `RDD` (a "big list") where EACH ENTRY is the result of applying the map function to a corresponding ENTRY of the input RDD.

## Lambda Function
- In general, the lambda function that is the parameter of `map` is applied to each element of the input RDD. Lambda function (in Python) has the format of
`lambda <parameter list>: <function body>`
The value returned by the lambda function is either 
- an explicit `return` statement, or
- the value returned by the last statement in the body of the function.

In [10]:
line_tokenized_RDD = text_RDD.map(lambda x: x.strip().split(" "))
line_tokenized_RDD

PythonRDD[5] at RDD at PythonRDD.scala:53

# Exercise 3 (5%)
Complete the code below to show the first five entries of the last RDD generated above.

In [11]:
line_tokenized_RDD.take(5)

[['“We',
  'can',
  'breathe',
  'a',
  'little',
  'more',
  'again.”',
  'Israelis',
  'rejoice',
  'at',
  'release',
  'of',
  'hostages'],
 ['From',
  'CNN’s',
  'Nadeen',
  'Ebrahim',
  'and',
  'Mike',
  'Schwartz',
  'in',
  'Tel',
  'Aviv'],
 [''],
 ['Israelis',
  'were',
  'elated',
  'following',
  'the',
  'release',
  'of',
  'three',
  'hostages',
  'from',
  'Gaza',
  'on',
  'Sunday,',
  'saying',
  'it',
  'almost',
  'feels',
  'unreal',
  'after',
  'so',
  'many',
  'months',
  'of',
  'waiting.'],
 ['']]

In [12]:
line1 = text_RDD.take(1)

In [13]:
line1

[' “We can breathe a little more again.” Israelis rejoice at release of hostages']

In [14]:
line1_string = line1[0]

In [15]:
line1_string

' “We can breathe a little more again.” Israelis rejoice at release of hostages'

In [16]:
line1_string_stripped = line1_string.strip()

In [17]:
line1_string_stripped

'“We can breathe a little more again.” Israelis rejoice at release of hostages'

# Exercise 3 (5%)
- Discuss the difference between the input string and output string to the strip() method above.

## Answer to Exercise 3:
- Strip removes the spaces at the beginning and end of a string. As you can tell in the line above, the beginning spaces between the quotations and we.

In [18]:
line1_string_stripped_split = line1_string.strip().split(" ")

In [19]:
line1_string_stripped_split

['“We',
 'can',
 'breathe',
 'a',
 'little',
 'more',
 'again.”',
 'Israelis',
 'rejoice',
 'at',
 'release',
 'of',
 'hostages']

# 6. Map (a transformation) 
- `textRDD.map(lambda ...)` applies the lambda expression, which strips and splits EACH entry of the input parameter, to EACH ENTRY of the input RDD.
- Each entry of the textRDD is a string representing a line of the input text file.
- Therefore, the output RDD of the `textRDD.map` statement is an RDD (big list) of sublists of words/tokens, one sublist for each line in the textRDD.

# Exercise 4 (10%)
(a) What is the result returned by the following code? (b) Explain the answer.

<pre>RDD = text_RDD.map(lambda x: x.strip().split(" "))
RDD.take(3)[1]</pre>

# Answer to Exercise 4:
- (a) ['From',
 'CNN’s',
 'Nadeen',
 'Ebrahim',
 'and',
 'Mike',
 'Schwartz',
 'in',
 'Tel',
 'Aviv']


- (b) You get this because first, map applies the lambda function that strips the spaces at beginning and end, and splits each string by the delimeter " ". Then you take 3 strings in the list, and output the one at the first index.

# 7. flatMap (a transformation)
The Spark `flatMap` method is (like map) also a transformation. It returns an RDD that removes the boundary between entries of the input RDD big list.
- Applying `flatMap` to `line_tokenized_RDD` removes the boundary of different lines in the input text.  In another word, it merges the list of tokens for each lines in the text into a gigantic list of tokens for the entire input document.
- Intuively, the effect of `flatMap` can be understood as **flattening** the internal structures of its input RDD.

## Results of flatMap:
- We no longer see the sublists of tokens in `line_tokenized_RDD` that contain the line structure of the input file.  
- Instead, all tokens become an element of a gigantic list (output RDD of flatMap).

In [20]:
tokens_RDD = line_tokenized_RDD.flatMap(lambda x: x)
tokens_RDD.take(30)

['“We',
 'can',
 'breathe',
 'a',
 'little',
 'more',
 'again.”',
 'Israelis',
 'rejoice',
 'at',
 'release',
 'of',
 'hostages',
 'From',
 'CNN’s',
 'Nadeen',
 'Ebrahim',
 'and',
 'Mike',
 'Schwartz',
 'in',
 'Tel',
 'Aviv',
 '',
 'Israelis',
 'were',
 'elated',
 'following',
 'the',
 'release']

# Exercise 5 (10%)
(a) What is the result returned by the following code? (b) Explain the answer.

<pre>RDD = text_RDD.flatmap(lambda x: x.strip().split(" "))
RDD.take(3)[1]</pre>

## Answers to Exercise 5:
- (a)'can'
- (b)The RDD takes the flatMap function, which put's everything in a one dimensional (flat) list and applies it to everything with lambda, again stripping the spaces at beginning and end, and splitting each word into its own string. We then take the first 3 strings, and then take the one at the first index.


# Part B: Counting Word Frequency using MapReduce

### We want to count the total number of time a word/token occurs in the input dataset. We can use the concept of MapReduce to do this in a "scalable" way such that we can do this calculation even if the size of dataset is too large to fit into a computer.

### A MapReduce way to achieve this involves two steps:
- Step 1: map each word into a key value pair 
`(<word>, 1)`
-- The key of this key-value pair is the word (in the input RDD); the value of the key-value pair is the number 1.
- Step 2: Use reduceByKey in Spark to aggregate all key-value pairs with the same key into a total count for each key/token in the format of 
`(<word>, <count>)`.

# Using map and reduceByKey for counting word frequency
- The first step above can be achieved in Sprk by mapping a the lambda function ``(lambda x: (x, 1))`` to the giantic (flattened) list (RDD) of tokens. 
- The second step above can be achieved in Spark using `reduceByKey`.
- In the example below, the lambda function returns the key value pair ``(<input word>, 1)`` because it is the only (hence also the last) statement in the body of the lambda
expression. Recall that lambda express returns the value returned by the last statement of its body or an explicit `return` statement.

In [21]:
token_1_RDD = tokens_RDD.map(lambda x: (x, 1))
token_1_RDD

PythonRDD[9] at RDD at PythonRDD.scala:53

In [22]:
token_1_RDD.take(10)

[('“We', 1),
 ('can', 1),
 ('breathe', 1),
 ('a', 1),
 ('little', 1),
 ('more', 1),
 ('again.”', 1),
 ('Israelis', 1),
 ('rejoice', 1),
 ('at', 1)]

# 8. reduceByKey (a transformation)

- Notice `reduceByKey` is not an action.  It is also a transformation.
- `reduceByKey` has TWO parameters.
- The first parameter of `reduceByKety` is a reduce/aggregation function, which is used to aggregate all key value pairs with the same key. 
- The second parameter of `reduceByKey` is the number of partitions used to partition the keys (so that the reduce task can be distributed among multiple reduce workers for scalability). In the cluster mode, the choice of this number should include considerations about the total number of nodes available in the cluster

# 9. Aggregation Function of reduceByKey
  
- The aggregation function `lambda x,y: x+y` has two parameters (x and y in the code below).
- The first parameter of the lambda expression (x) is an counter (initial value is 0).
- The second parameter of the lambda expression (y) is the value from the input key value pair.
- The result of `x+y` becomes the new counter, which is returned by the lambda function.
- Because reduceByKey aggregates key-value pairs with the same key, the updated counters returned by the lambda expression are added together to compute the total count. 
- Since the 'values' in the input key-value pairs are all 1, this lambda function simplies increment the accumulator for each occurance of the key.  Because reduceByKey aggregates across the entire input RDD, the final value (i.e., accumulator) for each word/token is the total number it occurs in the Twitter dataset.

# Exercise 6 (10 points) 
Complete the code below by (1) specifying the number for partitioning the keys to be 2, and (2) show the result of word count for a few words.

In [23]:
WF_RDD = token_1_RDD.reduceByKey(lambda x,y: x+y, 2)
WF_RDD.take(3)

[('“We', 4), ('more', 3), ('again.”', 1)]

# Exercise 7 (10 points) 
Complete the code below to save the word frequency in the output file (`Lab2_word_count.txt` under your Lab2 directory).

# saveAsTextFile (action)
- This action saves the content of the input RDD in the path specified.  
- Even though we add ".txt" extension to the directory (to remind ourself it contains text file, not CSV), the path is used as a directory to store all partitions of the RDD (i.e., `WF_RDD` in this lab). 

## **An important requirement for saveAsTextFile is the path for the output files SHOULD NOT EXIST.**  

## Debugging Tip #1: 
Before you run saveAsTextFile each time, double check whether the output path used does not already exist.  If it does, either 
- (a) remove the directory, or
- (b) change the output path to one that has not been used.

In [24]:
output_file = "/storage/home/ajv5723/work/Lab2/Lab2_word_count.txt"
WF_RDD.saveAsTextFile(output_file)

# Exercise 8 (10 points) Fill in the directory path with your correct PSUID (like what you did in Exercise 1 and Exercise 7.  Execute the code below. 
- If you notice the typo in the filename, do not fix it in this Exercise.
- (a) When do you expect to see the first error message before running the code below?
- (b) When (after exeucting which cell) does the error message actually show up?

In [25]:
text3_RDD = sc.textFile("/storage/home/ajv5723/work/Lab2/Hostagereleased.txt")
text3_RDD

/storage/home/ajv5723/work/Lab2/Hostagereleased.txt MapPartitionsRDD[20] at textFile at NativeMethodAccessorImpl.java:0

In [26]:
line3_RDD = text3_RDD.map(lambda line: line.strip().split(" "))

In [27]:
KV3_RDD = line3_RDD.flatMap(lambda x: x ).map(lambda x: (x, 1))

In [28]:
WC3_RDD = KV3_RDD.reduceByKey(lambda x,y: x+y, 2)

In [29]:
output_file2 = "/storage/home/ajv5723/work/Lab2/Lab2_word_count3.txt"
WC3_RDD.saveAsTextFile(output_file2)

25/01/23 13:13:35 ERROR SparkHadoopWriter: Aborting job job_202501231313354389856205588991182_0027.
org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/storage/home/ajv5723/work/Lab2/Hostagereleased.txt
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:205)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:300)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:296)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:300)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:296)
	at org.apache.spark.api

Py4JJavaError: An error occurred while calling o157.saveAsTextFile.
: org.apache.spark.SparkException: Job aborted.
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:106)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopDataset$1(PairRDDFunctions.scala:1090)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:414)
	at org.apache.spark.rdd.PairRDDFunctions.saveAsHadoopDataset(PairRDDFunctions.scala:1088)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopFile$4(PairRDDFunctions.scala:1061)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScop

# Answer to Exercise 8 (10 points):
### Type your answer to Exercise 8 below:
- (a) After the first cell, because it references "hostages released" which already exists
- (b)after executing this cell we get an error:
output_file2 = "/storage/home/ajv5723/work/Lab2/Lab2_word_count3.txt"
WC3_RDD.saveAsTextFile(output_file2)

In [30]:
sc.stop()

## The `.stop()` method terminates the given SparkContext. 